# Zadanie 7: automatyczne różniczkowanie

Termin realizacji: 26 maja 2025

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

Poniższy kod ilustruje definiowanie reguł automatycznego różniczkowania dla własnej funkcji realizującej operację potęgowania ($x^y$) która została zaimplementowan aby przyspieszyć liczenie wyniku dla przypadków gdy $y=0$ lub $y=1$. Naiwna implementacja tej optymalizacji (funkcja `my_pow`) powoduje jednak zwracanie błędnych pochodnych przez system automatycznego różniczkowania.
Problem można poprawić na jeden z dwóch sposobów:
1. Rozwinięcie w szereg Taylora w punktach odpowiadających szczególnym przypadkom (`my_pow_v2`). Jest to podejście działające niezależnie od użytego backendu automatycznego różniczkowania, jednak spowalnia działanie kodu.
2. Napisanie specjalnych reguł automatycznego różniczkowania dla naszej funkcji (`my_pow_v3`). Pokazane są przykłady realizacji tego podejścia w oparciu o biblioteki `ForwardDiff.jl` oraz `ChainRules.jl`.

## Na 3.0

Do realizacji:

1. Dokończ funkcję `my_pow_v2` rozwijając ją w szereg Taylora w $y=1$ analogicznie do rozwinięcia w $y=2$.
2. Porównaj szybkość działania `^`, `my_pow` i `my_pow_v2` we wszystkich trzech przypadkach ($y=0$, $y=1$, $y \notin \{0, 1\}$) oraz różnice w wartościach pochodnych pomiędzy standardową funkcją potęgowania (`^`) oraz funkcją `my_pow_v2` na podstawie skryptu z końca tego notatnika. Zanotuj bezwzględne wartości różnic między wyliczonymi wartościami pochodnych dla wszystkich rozważanych w teście przypadków.

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj porównanie z wartościami pochodnych wyliczonych w oparciu o bibliotekę `FiniteDifferences.jl`.
3. Porównaj szybkość wyliczania pochodnych w oparciu o biblioteki `ForwardDiff.jl`, `Zygote.jl` i `FiniteDifferences.jl` w rozważanych w skrypcie przypadkach.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0
2. Dokończ pisanie reguły automatycznego różniczkowania dla funkcji `my_pow_v3`. Większość reguły znajduje się we fragmencie zaczynającym się od `ForwardDiff.@define_binary_dual_op`. Brakujące fragmenty zostały w nim zastąpione rzucaniem błędu.


In [5]:

using ForwardDiff, Zygote
using BenchmarkTools

function my_pow(x::Real, y::Real)
    if y == 0
        return one(x)
    elseif y == 1
        return x
    else
        return x^y
    end
end


function my_pow_v2(x::Real, y::Real)
    if y == 0
        # rozwijamy w szereg Taylora w y = 0
        return one(x) + y * log(x)
    elseif y == 1
        # rozwijamy w szereg Taylora w y = 1
        return x + error("TODO")
    else
        return x^y
    end
end


my_pow_v2 (generic function with 1 method)

In [ ]:
function my_pow_v3(x::Real, y::Real)
    if y == 0
        return one(x)
    elseif y == 1
        return x
    else
        return x^y
    end
end

using ForwardDiff: value, partials, Dual, _mul_partials, isconstant

ForwardDiff.@define_binary_dual_op(
    my_pow_v3, # dwuargumentowa funkcja różniczkowana
    begin
        # kod wykonywany gdy różniczkujemy względem obu argumentów
        vx, vy = value(x), value(y)
        expv = my_pow_v3(vx, vy)
        powval = vy * my_pow_v3(vx, vy - 1)
        if isconstant(y)
            logval = one(expv)
        elseif iszero(vx) && vy > 0
            logval = zero(vx)
        else
            logval = expv * log(vx)
        end
        new_partials = _mul_partials(partials(x), partials(y), powval, logval)
        return Dual{Txy}(expv, new_partials)
    end,
    begin
        # kod wykonywany gdy różniczkujemy względem pierwszego argumentu
        v = value(x)
        expv = my_pow_v3(v, y)
        if y == zero(y) || iszero(partials(x))
            new_partials = zero(partials(x))
        else
            new_partials = error("TODO")
        end
        return Dual{Tx}(expv, new_partials)
    end,
    begin
        # kod wykonywany gdy różniczkujemy względem drugiego argumentu
        v = value(y)
        expv = my_pow_v3(x, v)
        deriv = error("TODO")
        return Dual{Ty}(expv, deriv * partials(y))
    end
)

my_pow_v3 (generic function with 15 methods)

In [7]:
using ChainRules

using ChainRules: ProjectTo, _pow_grad_x, _pow_grad_p, NoTangent

function ChainRules.rrule(::typeof(my_pow_v3), x::Real, p::Real)
    y = x^p
    project_x = ProjectTo(x)
    project_p = ProjectTo(p)
    function power_pullback(dy)
        _dx = _pow_grad_x(x, p, y)
        return (
            NoTangent(), 
            project_x(_dx * dy),
            project_p(_pow_grad_p(x, p, y) * dy)
        )
    end
    return y, power_pullback
end


In [8]:

function test()

    for (a, b) in [(10.0, 1.0), (10.0, 0.0), (10.0, 2.5)]
        println("\n>>> a = $a, b=$b\n")

        println(@btime $a^$b)
        println(@btime my_pow($a, $b))
        println(@btime my_pow_v2($a, $b))
        println(@btime my_pow_v3($a, $b))

        for diff in [ForwardDiff.derivative, Zygote.gradient]
            println("\nUsing differentiation through $(repr(diff))\n")
            println("d/dx x^y: ", diff(x -> x^b, a))
            println("d/dy x^y: ", diff(y -> a^y, b))
            println("d/dy x^x: ", diff(x -> x^x, b))

            println("d/dx my_pow(x, y): ", diff(x -> my_pow(x, b), a))
            println("d/dy my_pow(x, y): ", diff(y -> my_pow(a, y), b))
            println("d/dy my_pow(x, x): ", diff(x -> my_pow(x, x), b))

            println("d/dx my_pow_v2(x, y): ", diff(x -> my_pow_v2(x, b), a))
            println("d/dy my_pow_v2(x, y): ", diff(y -> my_pow_v2(a, y), b))
            println("d/dy my_pow_v2(x, x): ", diff(x -> my_pow_v2(x, x), b))

            println("d/dx my_pow_v3(x, y): ", diff(x -> my_pow_v3(x, b), a))
            println("d/dy my_pow_v3(x, y): ", diff(y -> my_pow_v3(a, y), b))
            println("d/dx my_pow_v3(x, x): ", diff(x -> my_pow_v3(x, x), b))
        end
    end
end

test()


>>> a = 10.0, b=1.0

  5.000 ns (0 allocations: 0 bytes)
10.0
  2.231 ns (0 allocations: 0 bytes)
10.0


ErrorException: TODO